In [ ]:
import csv
import os
import re
import random
import pickle
import warnings
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import sklearn
import seaborn as sns
import textwrap
import pickle
import matplotlib as mpl
from datetime import datetime
from matplotlib_venn import venn2
from scipy.stats import pearsonr
from typing import List, Dict, Optional

In [ ]:
# Define modality renaming dictionary (modality_map)
modality_map = {
'hearing': 'Hearing',
'immune': 'Immune',
'renalhepatic': 'Renal & Hepatic',
'metabolic': 'Metabolic',
'cardiopulmonary': 'Cardiopulmonary',
'musculoskeletal': 'Musculoskeletal',
'bone_densitometry': 'Bone Densitometry of Heel',
'pwa': 'Pulse Wave Analysis',
'heart_mri': 'Heart MRI',
'carotid_ultrasound': 'Carotid Ultrasound',
'arterial_stiffness': 'Arterial Stiffness',
'ecg_rest': 'ECG at Rest',
'body_composition_by_impedance': 'Body Composition by Impedance',
'body_composition_dxa': 'Body Composition by DXA',
'bone_dxa': 'Bone Size, Mineral and Density by DXA',
'kidneys_mri': 'Kidney MRI',
'liver_mri': 'Liver MRI',
'abdominal_composition_mri_18_vars': 'Abdominal Composition by MRI',
'abdominal_organ_composition_mri_13_vars': 'Abdominal Organ Composition by MRI',
'struct_fast' : 'Regional grey matter volumes (FSL FAST)',
'struct_sub_first': 'Subcortical volumes (FSL FIRST)',

'struct_fs_aseg_mean_intensity' : 'ASEG Mean Intensity',
'struct_fs_aseg_volume' : 'ASEG Volume',


'struct_ba_exvivo_area' : 'BA ex-vivo Area',
'struct_ba_exvivo_mean_thickness' : 'BA ex-vivo Mean Thickness',
'struct_ba_exvivo_volume' : 'BA ex-vivo Volume',

'struct_a2009s_area' : 'a2009s Area',
'struct_a2009s_mean_thickness' : 'a2009s Mean Thickness',
'struct_a2009s_volume' : 'a2009s Volume',


'struct_dkt_area' : 'Desikan-Killiany-Tourville Area',
'struct_dkt_mean_thickness' : 'Desikan-Killiany-Tourville Mean Thickness',
'struct_dkt_volume' : 'Desikan-Killiany-Tourville Volume',


'struct_desikan_gw' : 'Desikan Grey\White Matter Contrast',
'struct_desikan_pial' : 'Desikan Pial',

'struct_desikan_white_area' : 'Desikan White Matter Area',
'struct_desikan_white_mean_thickness' : 'Desikan White Matter Mean Thickness',
'struct_desikan_white_volume' : 'Desikan White Matter Volume',
"struct_subsegmentation":'Subcortical Volumetric Subsegmentation',

'add_t1' : 'Whole-Brain T1w',
'add_t2' : 'Whole-Brain T2w',
"dwi_FA_tbss": "FA TBSS",
"dwi_FA_prob": "FA Probabilistic",
"dwi_MD_tbss": "MD TBSS",
"dwi_MD_prob": "MD Probabilistic",
"dwi_L1_tbss": "L1 TBSS",
"dwi_L1_prob": "L1 Probabilistic",
"dwi_L2_tbss": "L2 TBSS",
"dwi_L2_prob": "L2 Probabilistic",
"dwi_L3_tbss": "L3 TBSS",
"dwi_L3_prob": "L3 Probabilistic",
"dwi_MO_tbss": "MO TBSS",
"dwi_MO_prob": "MO Probabilistic",
"dwi_OD_tbss": "OD TBSS",
"dwi_OD_prob": "OD Probabilistic",
"dwi_ICVF_tbss": "ICVF TBSS",
"dwi_ICVF_prob": "ICVF Probabilistic",
"dwi_ISOVF_tbss": "ISOVF TBSS",
"dwi_ISOVF_prob": 'ISOVF Probabilistic',
"amplitudes_21": " 21 IC Amplitudes",
"amplitudes_55": "55 IC Amplitudes",
"full_correlation_21": "21 IC Full Correlation",
"full_correlation_55": "55 IC Full Correlation",
"partial_correlation_21": " 21 IC Partial Correlation",
"partial_correlation_55": " 55 IC Partial Correlation",
# aparc Tian S1 (I)
'aparc_Tian_S1_FA_i2': 'aparc-I FA',
'aparc_Tian_S1_Length_i2': 'aparc-I Length',
'aparc_Tian_S1_SIFT2_FBC_i2': 'aparc-I SIFT2 FBC',
'aparc_Tian_S1_Streamline_Count_i2': 'aparc-I Streamline Count',

# aparc a2009s Tian S1 (I)
'aparc_a2009s_Tian_S1_FA_i2': 'aparc.a2009s-I FA',
'aparc_a2009s_Tian_S1_Length_i2': 'aparc.a2009s-I Length',
'aparc_a2009s_Tian_S1_SIFT2_FBC_i2': 'aparc.a2009s-I SIFT2 FBC',
'aparc_a2009s_Tian_S1_Streamline_Count_i2': 'aparc.a2009s-I Streamline Count',

# Glasser Tian S1 (I)
'Glasser_Tian_S1_FA_i2': 'Glasser-I FA',
'Glasser_Tian_S1_Length_i2': 'Glasser-I Length',
'Glasser_Tian_S1_SIFT2_FBC_i2': 'Glasser-I SIFT2 FBC',
'Glasser_Tian_S1_Streamline_Count_i2': 'Glasser-I Streamline Count',

# Glasser Tian S4 (IV)
'Glasser_Tian_S4_FA_i2': 'Glasser-IV FA',
'Glasser_Tian_S4_Length_i2': 'Glasser-IV Length',
'Glasser_Tian_S4_SIFT2_FBC_i2': 'Glasser-IV SIFT2 FBC',
'Glasser_Tian_S4_Streamline_Count_i2': 'Glasser-IV Streamline Count',

# Schaefer7n1000p Tian S4 (IV) (in reality: Schaefer7n200p Tian S1)
'Schaefer7n1000p_Tian_S4_FA_i2': 'Schaefer7n200p-I FA', #'Schaefer7n1000p-IV FA',
'Schaefer7n1000p_Tian_S4_Length_i2': 'Schaefer7n200p-I Length',#'Schaefer7n1000p-IV Length',
'Schaefer7n1000p_Tian_S4_SIFT2_FBC_i2': 'Schaefer7n200p-I SIFT2 FBC',#'Schaefer7n1000p-IV SIFT2 FBC',
'Schaefer7n1000p_Tian_S4_Streamline_Count_i2': 'Schaefer7n200p-I Streamline Count', #'Schaefer7n1000p-IV Streamline Count'

# Schaefer7n200p Tian S4 (IV) (in reality: Schaefer7n500p Tian S4)
'Schaefer7n200p_Tian_S1_FA_i2': 'Schaefer7n500p-IV FA',
'Schaefer7n200p_Tian_S1_Length_i2': 'Schaefer7n500p-IV Length',
'Schaefer7n200p_Tian_S1_SIFT2_FBC_i2': 'Schaefer7n500p-IV SIFT2 FBC',
'Schaefer7n200p_Tian_S1_Streamline_Count_i2': 'Schaefer7n500p-IV Streamline Count',

# Schaefer7n500p Tian S4 (IV) (in reality: Schaefer7n1000p Tian S4)
'Schaefer7n500p_Tian_S4_FA_i2': 'Schaefer7n1000p-IV FA',
'Schaefer7n500p_Tian_S4_Length_i2': 'Schaefer7n1000p-IV Length',
'Schaefer7n500p_Tian_S4_SIFT2_FBC_i2': 'Schaefer7n1000p-IV SIFT2 FBC',
'Schaefer7n500p_Tian_S4_Streamline_Count_i2': 'Schaefer7n1000p-IV Streamline Count',

# Resting state 
'full_correlation_aparc_a2009s_Tian_S1' : 'aparc.a2009s-I Full Correlation',
'full_correlation_aparc_Tian_S1': 'aparc-I Full Correlation',
'full_correlation_Glasser_Tian_S1': 'Glasser-I Full Correlation',
'full_correlation_Glasser_Tian_S4': 'Glasser-IV Full Correlation',
'full_correlation_Schaefer7n200p_Tian_S1': 'Schaefer7n200p-I Full Correlation',
'full_correlation_Schaefer7n500p_Tian_S4': 'Schaefer7n500p-IV Full Correlation',
'partial_correlation_aparc_a2009s_Tian_S1': 'aparc.a2009s-I Partial Correlation',
'partial_correlation_aparc_Tian_S1': 'aparc-I Partial Correlation',
'partial_correlation_Glasser_Tian_S1': 'Glasser-I Partial Correlation',
'partial_correlation_Glasser_Tian_S4': 'Glasser-IV Partial Correlation',
'partial_correlation_Schaefer7n200p_Tian_S1': 'Schaefer7n200p-I Partial Correlation',
'partial_correlation_Schaefer7n500p_Tian_S4': 'Schaefer7n500p-IV Partial Correlation',

# Individual behavioural
'activity_touchscreen' :'Daily activity',
'accelerometry':'Accelerometry',
'activity_MET':'Metabolic equivalent of task',
'activity_byrecall_i3':'Yesterday activity',
'sleep_ac':'Sleep',
'localenv':'Local environment',
'diet':'Diet',
'electronic_device_use':'Electronic device use',
'alcohol':'Alcohol',
'sun_i0':'Sun exposure',
'sexual_factors':'Sexual factors',
'smoking':'Smoking',

'allmri': 'Composite Brain',
'dwi': 'dwMRI',
'smri': 'sMRI',
'rs': 'rsMRI',
'body': 'Composite Body',
'lifestyle-envir': 'Lifestyle and Environment',
'brain-plus-body': 'Brain and Body',
'brain-body': 'Brain and Body',
'body-only': 'Composite Body' ,
'all': 'Lifestyle, Environment, Body, and Brain'
}

# Define modality names for renaming
modality_names = {
'hearing': 'Hearing',
'immune': 'Immune',
'renalhepatic': 'Renal & Hepatic',
'metabolic': 'Metabolic',
'cardiopulmonary': 'Cardiopulmonary',
'musculoskeletal': 'Musculoskeletal',
'bone_densitometry': 'Bone Densitometry of Heel',
'pwa': 'Pulse Wave Analysis',
'heart_mri': 'Heart MRI',
'carotid_ultrasound': 'Carotid Ultrasound',
'arterial_stiffness': 'Arterial Stiffness',
'ecg_rest': 'ECG at Rest',
'body_composition_by_impedance': 'Body Composition by Impedance',
'body_composition_dxa': 'Body Composition by DXA',
'bone_dxa': 'Bone Size, Mineral and Density by DXA',
'kidneys_mri': 'Kidney MRI',
'liver_mri': 'Liver MRI',
'abdominal_composition_mri_18_vars': 'Abdominal Composition by MRI',
'abdominal_organ_composition_mri_13_vars': 'Abdominal Organ Composition by MRI',
'struct_fast' : 'Regional grey matter volumes (FSL FAST)',
'struct_sub_first': 'Subcortical volumes (FSL FIRST)',

'struct_fs_aseg_mean_intensity' : 'ASEG Mean Intensity',
'struct_fs_aseg_volume' : 'ASEG Volume',

'struct_ba_exvivo_area' : 'BA ex-vivo Area',
'struct_ba_exvivo_mean_thickness' : 'BA ex-vivo Mean Thickness',
'struct_ba_exvivo_volume' : 'BA ex-vivo Volume',

'struct_a2009s_area' : 'a2009s Area',
'struct_a2009s_mean_thickness' : 'a2009s Mean Thickness',
'struct_a2009s_volume' : 'a2009s Volume',


'struct_dkt_area' : 'Desikan-Killiany-Tourville Area',
'struct_dkt_mean_thickness' : 'Desikan-Killiany-Tourville Mean Thickness',
'struct_dkt_volume' : 'Desikan-Killiany-Tourville Volume',


'struct_desikan_gw' : 'Desikan Grey\White Matter Contrast',
'struct_desikan_pial' : 'Desikan Pial',

'struct_desikan_white_area' : 'Desikan White Matter Area',
'struct_desikan_white_mean_thickness' : 'Desikan White Matter Mean Thickness',
'struct_desikan_white_volume' : 'Desikan White Matter Volume',
"struct_subsegmentation":'Subcortical Volumetric Subsegmentation',

'add_t1' : 'Whole-Brain T1w',
'add_t2' : 'Whole-Brain T2w',
"dwi_FA_tbss": "FA TBSS",
"dwi_FA_prob": "FA Probabilistic",
"dwi_MD_tbss": "MD TBSS",
"dwi_MD_prob": "MD Probabilistic",
"dwi_L1_tbss": "L1 TBSS",
"dwi_L1_prob": "L1 Probabilistic",
"dwi_L2_tbss": "L2 TBSS",
"dwi_L2_prob": "L2 Probabilistic",
"dwi_L3_tbss": "L3 TBSS",
"dwi_L3_prob": "L3 Probabilistic",
"dwi_MO_tbss": "MO TBSS",
"dwi_MO_prob": "MO Probabilistic",
"dwi_OD_tbss": "OD TBSS",
"dwi_OD_prob": "OD Probabilistic",
"dwi_ICVF_tbss": "ICVF TBSS",
"dwi_ICVF_prob": "ICVF Probabilistic",
"dwi_ISOVF_tbss": "ISOVF TBSS",
"dwi_ISOVF_prob": 'ISOVF Probabilistic',
"amplitudes_21": " 21 IC Amplitudes",
"amplitudes_55": "55 IC Amplitudes",
"full_correlation_21": "21 IC Full Correlation",
"full_correlation_55": "55 IC Full Correlation",
"partial_correlation_21": " 21 IC Partial Correlation",
"partial_correlation_55": " 55 IC Partial Correlation",
# aparc Tian S1 (I)
'aparc_Tian_S1_FA_i2': 'aparc-I FA',
'aparc_Tian_S1_Length_i2': 'aparc-I Length',
'aparc_Tian_S1_SIFT2_FBC_i2': 'aparc-I SIFT2 FBC',
'aparc_Tian_S1_Streamline_Count_i2': 'aparc-I Streamline Count',

# aparc a2009s Tian S1 (I)
'aparc_a2009s_Tian_S1_FA_i2': 'aparc.a2009s-I FA',
'aparc_a2009s_Tian_S1_Length_i2': 'aparc.a2009s-I Length',
'aparc_a2009s_Tian_S1_SIFT2_FBC_i2': 'aparc.a2009s-I SIFT2 FBC',
'aparc_a2009s_Tian_S1_Streamline_Count_i2': 'aparc.a2009s-I Streamline Count',

# Glasser Tian S1 (I)
'Glasser_Tian_S1_FA_i2': 'Glasser-I FA',
'Glasser_Tian_S1_Length_i2': 'Glasser-I Length',
'Glasser_Tian_S1_SIFT2_FBC_i2': 'Glasser-I SIFT2 FBC',
'Glasser_Tian_S1_Streamline_Count_i2': 'Glasser-I Streamline Count',

# Glasser Tian S4 (IV)
'Glasser_Tian_S4_FA_i2': 'Glasser-IV FA',
'Glasser_Tian_S4_Length_i2': 'Glasser-IV Length',
'Glasser_Tian_S4_SIFT2_FBC_i2': 'Glasser-IV SIFT2 FBC',
'Glasser_Tian_S4_Streamline_Count_i2': 'Glasser-IV Streamline Count',

# Schaefer7n1000p Tian S4 (IV) (in reality: Schaefer7n200p Tian S1)
'Schaefer7n1000p_Tian_S4_FA_i2': 'Schaefer7n200p-I FA', #'Schaefer7n1000p-IV FA',
'Schaefer7n1000p_Tian_S4_Length_i2': 'Schaefer7n200p-I Length',#'Schaefer7n1000p-IV Length',
'Schaefer7n1000p_Tian_S4_SIFT2_FBC_i2': 'Schaefer7n200p-I SIFT2 FBC',#'Schaefer7n1000p-IV SIFT2 FBC',
'Schaefer7n1000p_Tian_S4_Streamline_Count_i2': 'Schaefer7n200p-I Streamline Count', #'Schaefer7n1000p-IV Streamline Count'

# Schaefer7n200p Tian S4 (IV) (in reality: Schaefer7n500p Tian S4)
'Schaefer7n200p_Tian_S1_FA_i2': 'Schaefer7n500p-IV FA',
'Schaefer7n200p_Tian_S1_Length_i2': 'Schaefer7n500p-IV Length',
'Schaefer7n200p_Tian_S1_SIFT2_FBC_i2': 'Schaefer7n500p-IV SIFT2 FBC',
'Schaefer7n200p_Tian_S1_Streamline_Count_i2': 'Schaefer7n500p-IV Streamline Count',

# Schaefer7n500p Tian S4 (IV) (in reality: Schaefer7n1000p Tian S4)
'Schaefer7n500p_Tian_S4_FA_i2': 'Schaefer7n1000p-IV FA',
'Schaefer7n500p_Tian_S4_Length_i2': 'Schaefer7n1000p-IV Length',
'Schaefer7n500p_Tian_S4_SIFT2_FBC_i2': 'Schaefer7n1000p-IV SIFT2 FBC',
'Schaefer7n500p_Tian_S4_Streamline_Count_i2': 'Schaefer7n1000p-IV Streamline Count',

# Resting state 
'full_correlation_aparc_a2009s_Tian_S1' : 'aparc.a2009s-I Full Correlation',
'full_correlation_aparc_Tian_S1': 'aparc-I Full Correlation',
'full_correlation_Glasser_Tian_S1': 'Glasser-I Full Correlation',
'full_correlation_Glasser_Tian_S4': 'Glasser-IV Full Correlation',
'full_correlation_Schaefer7n200p_Tian_S1': 'Schaefer7n200p-I Full Correlation',
'full_correlation_Schaefer7n500p_Tian_S4': 'Schaefer7n500p-IV Full Correlation',
'partial_correlation_aparc_a2009s_Tian_S1': 'aparc.a2009s-I Partial Correlation',
'partial_correlation_aparc_Tian_S1': 'aparc-I Partial Correlation',
'partial_correlation_Glasser_Tian_S1': 'Glasser-I Partial Correlation',
'partial_correlation_Glasser_Tian_S4': 'Glasser-IV Partial Correlation',
'partial_correlation_Schaefer7n200p_Tian_S1': 'Schaefer7n200p-I Partial Correlation',
'partial_correlation_Schaefer7n500p_Tian_S4': 'Schaefer7n500p-IV Partial Correlation',

# Individual behavioural
'activity_touchscreen' :'Daily activity',
'accelerometry':'Accelerometry',
'activity_MET':'Metabolic equivalent of task',
'activity_byrecall_i3':'Yesterday activity',
'sleep_ac':'Sleep',
'localenv':'Local environment',
'diet':'Diet',
'electronic_device_use':'Electronic device use',
'alcohol':'Alcohol',
'sun_i0':'Sun exposure',
'sexual_factors':'Sexual factors',
'smoking':'Smoking',

'allmri': 'Composite Brain',
'dwi': 'dwMRI',
'smri': 'sMRI',
'rs': 'rsMRI',
'body': 'Composite Body',
'lifestyle-envir': 'Lifestyle and Environment',
'brain-plus-body': 'Brain and Body',
'brain-body': 'Brain and Body',
'body-only': 'Composite Body' ,
'all': 'Lifestyle, Environment, Body, and Brain'
}

In [ ]:
# Define modalities
modalities_body = [
'immune',
'renalhepatic',
'metabolic',
'cardiopulmonary',
'musculoskeletal',
'bone_densitometry',
'pwa',
'heart_mri',
'carotid_ultrasound',
'arterial_stiffness',
'ecg_rest',
'body_composition_by_impedance',
'body_composition_dxa',
'bone_dxa',
'kidneys_mri',
'liver_mri',
'abdominal_composition_mri_18_vars', #17 vars
'abdominal_organ_composition_mri_13_vars', #12 vars
'hearing'
]

modalities_brain = [
'struct_fast',
'struct_sub_first',
'struct_fs_aseg_mean_intensity',
'struct_fs_aseg_volume',
'struct_ba_exvivo_area', 
'struct_ba_exvivo_mean_thickness',
'struct_ba_exvivo_volume',
'struct_a2009s_area',
'struct_a2009s_mean_thickness',
'struct_a2009s_volume',
'struct_dkt_area',
'struct_dkt_mean_thickness',
'struct_dkt_volume',
'struct_desikan_gw',
'struct_desikan_pial',
'struct_desikan_white_area',
'struct_desikan_white_mean_thickness',
'struct_desikan_white_volume',
'struct_subsegmentation',
'add_t1',
'add_t2',

"dwi_FA_tbss", "dwi_FA_prob",
"dwi_MD_tbss", "dwi_MD_prob",
"dwi_L1_tbss", "dwi_L1_prob",
"dwi_L2_tbss", "dwi_L2_prob",
"dwi_L3_tbss", "dwi_L3_prob",
"dwi_MO_tbss", "dwi_MO_prob",
"dwi_OD_tbss", "dwi_OD_prob",
"dwi_ICVF_tbss", "dwi_ICVF_prob",
"dwi_ISOVF_tbss", "dwi_ISOVF_prob",

'aparc_Tian_S1_FA_i2',
'aparc_Tian_S1_Length_i2',
'aparc_Tian_S1_SIFT2_FBC_i2',
'aparc_Tian_S1_Streamline_Count_i2',

'aparc_a2009s_Tian_S1_FA_i2',
'aparc_a2009s_Tian_S1_Length_i2',
'aparc_a2009s_Tian_S1_SIFT2_FBC_i2',
'aparc_a2009s_Tian_S1_Streamline_Count_i2',

'Glasser_Tian_S1_FA_i2',
'Glasser_Tian_S1_Length_i2',
'Glasser_Tian_S1_SIFT2_FBC_i2',
'Glasser_Tian_S1_Streamline_Count_i2',

'Glasser_Tian_S4_FA_i2',
'Glasser_Tian_S4_Length_i2',
'Glasser_Tian_S4_SIFT2_FBC_i2',
'Glasser_Tian_S4_Streamline_Count_i2',

'Schaefer7n200p_Tian_S1_FA_i2',
'Schaefer7n200p_Tian_S1_Length_i2',
'Schaefer7n200p_Tian_S1_SIFT2_FBC_i2',
'Schaefer7n200p_Tian_S1_Streamline_Count_i2',

'Schaefer7n1000p_Tian_S4_FA_i2',
'Schaefer7n1000p_Tian_S4_Length_i2',
'Schaefer7n1000p_Tian_S4_SIFT2_FBC_i2',
'Schaefer7n1000p_Tian_S4_Streamline_Count_i2',

"amplitudes_21",
"full_correlation_21",
"partial_correlation_21",
"amplitudes_55",
"full_correlation_55",
"partial_correlation_55",
'full_correlation_aparc_a2009s_Tian_S1',
'full_correlation_aparc_Tian_S1',
'full_correlation_Glasser_Tian_S1',
'full_correlation_Glasser_Tian_S4',
'full_correlation_Schaefer7n200p_Tian_S1',
'full_correlation_Schaefer7n500p_Tian_S4',
'partial_correlation_aparc_a2009s_Tian_S1',
'partial_correlation_aparc_Tian_S1',
'partial_correlation_Glasser_Tian_S1',
'partial_correlation_Glasser_Tian_S4',
'partial_correlation_Schaefer7n200p_Tian_S1',
'partial_correlation_Schaefer7n500p_Tian_S4'
]

modalities_behav = [
'activity_touchscreen',
'accelerometry',
'activity_MET',
'activity_byrecall_i3',
'sleep_ac',
'localenv',
'diet',
'electronic_device_use',
'alcohol',
'sun_i0',
'sexual_factors',
'smoking'
]

In [ ]:
# Configuration
warnings.simplefilter(action='ignore', category=FutureWarning)
from datetime import datetime
folds = range(0, 5)
base_path = r'Z:\UK_BB\brainbody'
fig_path = r'Z:\UK_BB\brainbody\figures\behav'

# Demographics confounds
demo = pd.read_csv(r'Z:\UK_BB\brainbody\demographics_full.csv')
# Rename columns and count NAs
df_demo_i2 = demo[[
'eid',
'31-0.0',
'21000-0.0',
'21003-2.0',]]
demo_full = df_demo_i2.rename(columns={
'31-0.0':'Sex',
'21000-0.0':'Ethnicity',
'21003-2.0':'Age',
})

age = demo_full['Age']
#/media/hcs-sci-psy-narun
sex = demo_full['Sex']

# Define a function that renamed columns based on modality map
def clean_modality_name(name, prefix_to_remove=None):

    clean_name = name
    
    # Remove specified prefix(es) if provided
    if prefix_to_remove is not None:
        if isinstance(prefix_to_remove, str):
            # Single prefix
            if clean_name.startswith(prefix_to_remove):
                clean_name = clean_name.replace(prefix_to_remove, '', 1)
        elif isinstance(prefix_to_remove, list):
            # Multiple prefixes - remove each one that matches
            for prefix in prefix_to_remove:
                if clean_name.startswith(prefix):
                    clean_name = clean_name.replace(prefix, '', 1)
                    break  # Remove only the first matching prefix
    
    # Use modality_map for display name, fallback to original if not found
    return modality_map.get(clean_name, clean_name)

### Compute correlation between g-factors predicted from each domain with composite marker

*g pred stack CORR g pred each behav domain*

In [ ]:
# Combine g-factors predicted from the stacked body model across five folds
g_pred_stacked = []
for fold in folds:
    try:
        stacked_path = os.path.join(
            base_path,
            'stacking',
            'lifestyle-envir',
            'folds',
            f'fold_{fold}',
            'g_pred',
            f'lifestyle-envir_target_pred_2nd_level_0_outer_test_fold_{fold}.csv'
        )
        df = pd.read_csv(stacked_path)
        g_pred_stacked.append(df)
    except Exception as e:
        print(f"Error loading stacked g-factor for fold {fold}: {str(e)}")

# Combine stacked predictions
g_pred_stacked = pd.concat(g_pred_stacked, axis=0, ignore_index=True)

# Merge modality-specific g-factor predictions
modality_frames = []
for modality in modalities_behav:
    modality_data = []
    for fold in folds:
        try:
            modality_path = os.path.join(
                base_path,
                'lifestyle-envir-body',
                'folds',
                f'fold_{fold}',
                'g_pred',
                f'{modality}_g_pred_XGB_test_with_id_fold_{fold}.csv'
            )
            df = pd.read_csv(modality_path)
            modality_data.append(df[['eid', f'g_pred_test_{modality}']])
        except Exception as e:
            print(f"Error loading {modality} for fold {fold}: {str(e)}")
    if modality_data:
        combined_modality = pd.concat(modality_data, axis=0, ignore_index=True)
        modality_frames.append(combined_modality)

# Start with the first modality DataFrame
merged_modalities = modality_frames[0]
# Merge the rest one by one
for df in modality_frames[1:]:
    merged_modalities = pd.merge(merged_modalities, df, on='eid', how='outer')

# Merge with stacked g-factor predictions
combined = pd.merge(merged_modalities, g_pred_stacked[['eid', f'g_pred_stack_test']], on='eid', how='inner')
combined.to_csv(os.path.join(base_path, 'feature_imp', 'feature_imp_behav', 'domains', f'g_pred_from_behav_mod_g_pred_stack_combined.csv'), index=False)
print(combined.shape)

In [ ]:
# Compute correlations
correlations = []
for col in combined.columns:
    if col not in ['eid', 'g_pred_stack_test']:
        try:
            x = combined['g_pred_stack_test']
            y = combined[col]
            mask = ~x.isna() & ~y.isna()
            if mask.sum() >= 2:
                r, p = stats.pearsonr(x[mask], y[mask])
            else:
                r, p = np.nan, np.nan

            correlations.append({
                'Modality': col,
                'Pearson r': r,
                'p_raw': p     # numeric p-value
            })

        except Exception as e:
            print(f"Error correlating {col}: {str(e)}")
            correlations.append({'Modality': col, 'Pearson r': np.nan, 'p_raw': np.nan})

# Convert to DataFrame
corr_results = pd.DataFrame(correlations)

# Bonferroni correction
n_tests = corr_results['p_raw'].notna().sum()
corr_results['p_bonf'] = corr_results['p_raw'] * n_tests
corr_results['p_bonf'] = corr_results['p_bonf'].clip(upper=1)
                                                                   
# Format p-values
corr_results['p_raw_formatted'] = corr_results['p_raw'].apply(
    lambda p: f"{p:.2e}" if pd.notna(p) else np.nan
)

corr_results['Bonferroni-corrected p-value'] = corr_results['p_bonf'].apply(
    lambda p: "<0.0001" if (pd.notna(p) and p < 0.0001) else (f"{p:.3f}" if pd.notna(p) else np.nan)
)

# Save to CSV
corr_results.sort_values('Pearson r', ascending=False).reset_index(drop=True).to_csv(os.path.join(base_path, 'feature_imp', 'feature_imp_behav', 'domains', 'g_pred_from_behav_mod_g_pred_stack_correlations.csv'), index=False)

# Apply renaming function before saving to Excel
corr_results['Modality'] = corr_results['Modality'].apply(
    lambda x: clean_modality_name(x, prefix_to_remove='g_pred_test_')
)

# Save to Excel
corr_results.sort_values('Pearson r', ascending=False).reset_index(drop=True).to_excel(
    os.path.join(base_path, 'feature_imp', 'feature_imp_behav', 'domains', 'g_pred_from_behav_mod_g_pred_stack_correlations.xlsx'),
    index=False,
    engine='openpyxl'
)

print("Correlation analysis complete. Results saved to CSV and Excel.")

### Correlation between behavioural features and composite behavioural marker

*g pred behav stack CORR scaled behav features*

In [ ]:
# Combine g-factors predicted from the stacked body model across five folds
g_pred_stacked = []

modality_domains = {
    'activity': ['activity_touchscreen', 'accelerometry', 'activity_MET', 'activity_byrecall_i3'],
    'envir': ['localenv'],
    'lifestyle': ['diet', 'electronic_device_use', 'alcohol', 'sun_i0', 'sexual_factors', 'smoking'],
    'sleep': ['sleep_ac']
}

for fold in folds:
    try:
        stacked_path = os.path.join(
            base_path,
            'stacking/lifestyle-envir',
            'folds',
            f'fold_{fold}',
            'g_pred',
            f'lifestyle-envir_target_pred_2nd_level_0_outer_test_fold_{fold}.csv'
        )
        df = pd.read_csv(stacked_path)
        g_pred_stacked.append(df)
    except Exception as e:
        print(f"Error loading stacked g-factor for fold {fold}: {str(e)}")

# Combine stacked predictions
g_pred_stacked = pd.concat(g_pred_stacked, axis=0, ignore_index=True)

# Pool scaled features and merge with eid from pre-scaled files
feature_to_modality = {}
all_features = []

for modality in modalities_behav:
    modality_features = []

    for fold in folds:
        try:
            for domain, modality_list in modality_domains.items():
                if modality in modality_list:
                    scaled_path = os.path.join(
                        base_path, domain, 'folds', f'fold_{fold}', 'scaling',
                        f'{modality}_test_scaled_fold_{fold}.csv'
                    )
                    prescaled_path = os.path.join(
                        base_path, domain, 'folds', f'fold_{fold}', 'suppl',
                        f'{modality}_test_feat_targ_fold_{fold}.csv'
                    )

            scaled_df = pd.read_csv(scaled_path)
            prescaled_df = pd.read_csv(prescaled_path)

            # Map features to domains
            for feature in scaled_df.columns:
                feature_to_modality[feature] = modality

            merged_df = pd.concat([prescaled_df[['eid']], scaled_df], axis=1)
            modality_features.append(merged_df)

        except Exception as e:
            print(f"Error processing {modality}, fold {fold}: {str(e)}")

    if modality_features:
        combined_modality = pd.concat(modality_features, axis=0, ignore_index=True)
        all_features.append(combined_modality)
        
# Merge all modalities column-wise on 'eid'
features_all_modalities = all_features[0]
for df in all_features[1:]:
    features_all_modalities = pd.merge(features_all_modalities, df, on='eid', how='outer')

print(f"Total columns in features_all_modalities: {len(features_all_modalities.columns)}")

# Merge features with stacked g-factor predictions on 'eid'
combined = pd.merge(features_all_modalities, g_pred_stacked[['eid', 'g_pred_stack_test']], on='eid', how='inner')

# Save the final combined DataFrame
output_path = os.path.join(base_path, 'feature_imp', 'feature_imp_behav', 'features', f'behav_features_g_pred_stack_combined.csv')
combined.to_csv(output_path, index=False)

print(f"Final combined shape: {combined.shape}")

In [ ]:
# Compute correlations with stacked g-factor
correlations = []
for col in combined.columns:
    if col not in ['eid', 'g_pred_stack_test']:
        try:
            x = combined['g_pred_stack_test']
            y = combined[col]
            mask = ~x.isna() & ~y.isna()
            if mask.sum() >= 2:
                r, p = stats.pearsonr(x[mask], y[mask])
            else:
                r, p = np.nan, np.nan

            correlations.append({
                'Phenotype': col,
                'Pearson r': r,
                'p_raw': p      # numeric p-value
            })

        except Exception as e:
            print(f"Error correlating {col}: {str(e)}")
            correlations.append({'Phenotype': col, 'Pearson r': np.nan, 'p_raw': np.nan})

# Convert to DataFrame
corr_results_orig = pd.DataFrame(correlations).reset_index(drop=True)

# Bonferroni correction
n_tests = corr_results_orig['p_raw'].notna().sum()
corr_results_orig['p_bonf'] = corr_results_orig['p_raw'] * n_tests
corr_results_orig['p_bonf'] = corr_results_orig['p_bonf'].clip(upper=1)

# Format
corr_results_orig['p_raw_formatted'] = corr_results_orig['p_raw'].apply(
    lambda p: f"{p:.2e}" if pd.notna(p) else ""
)

# Bonferroni p-values thresholded
corr_results_orig['Bonferroni-corrected p-value'] = corr_results_orig['p_bonf'].apply(
    lambda p: "<0.0001" if (pd.notna(p) and p < 0.0001)
              else (f"{p:.3f}" if pd.notna(p) else "")
)

# Sorted version (Pearson r)
corr_results_pooled_sorted = corr_results_orig.sort_values(
    'Pearson r', ascending=False
).reset_index(drop=True)


# Save to CSV
corr_results_orig.to_csv(os.path.join(base_path, 'feature_imp', 'feature_imp_behav', 'features', 'behav_features_g_pred_stack_correlations_not_sorted.csv'), index=False)
corr_results_pooled_sorted.to_csv(os.path.join(base_path, 'feature_imp', 'feature_imp_behav', 'features', 'behav_features_g_pred_stack_correlations_pooled_sorted.csv'), index=False)

# Apply renaming function before saving to Excel
corr_results_orig['Phenotype'] = corr_results_orig['Phenotype'].apply(
    lambda x: clean_modality_name(x, prefix_to_remove='g_pred_test_')
)

# Save to Excel
corr_results_orig.to_excel(
    os.path.join(base_path, 'feature_imp', 'feature_imp_behav', 'features', 'behav_features_g_pred_stack_correlations_not_sorted.xlsx'),
    index=False,
    engine='openpyxl'
)

corr_results_pooled_sorted.to_excel(
    os.path.join(base_path, 'feature_imp', 'feature_imp_behav', 'features', 'behav_features_g_pred_stack_correlations_pooled_sorted.xlsx'),
    index=False,
    engine='openpyxl'
)

print("Correlation analysis complete. Results saved to CSV and Excel.")

In [ ]:
# Sort correlations within each domain (modality)
corr_results_orig['Domain'] = corr_results_orig['Phenotype'].apply(
    lambda x: feature_to_modality.get(x, 'Unknown')
)

# Map domains
modality_display = {
    'activity_touchscreen': 'Activity: Daily behaviour',
    'diet': 'Diet',
    'accelerometry': 'Activity: Accelerometry',
    'sun_i0': 'Sun exposure',
    'activity_MET': 'Activity: MET',
    'localenv': 'Local environment',
    'sleep_ac': 'Sleep',
    'alcohol': 'Alcohol',
    'smoking': 'Smoking',
    'electronic_device_use': 'Electronic device use',
    'sexual_factors': 'Sexual Factors',
    'activity_byrecall_i3': 'Activity: Online (by recall)'
}

# Add display name column
corr_results_orig['Domain_renamed'] = corr_results_orig['Domain'].apply(
    lambda x: modality_display.get(x, 'Unknown')
)

# Desired order of display names
modality_order = [
    'Activity: Daily behaviour',
    'Diet',
    'Activity: Accelerometry',
    'Sun exposure',
    'Activity: MET',
    'Local environment',
    'Sleep',
    'Alcohol',
    'Smoking',
    'Electronic device use',
    'Sexual Factors',
    'Activity: Online (by recall)'
]

# Create rank for sorting
modality_rank = {name: i for i, name in enumerate(modality_order)}

# Add rank column
corr_results_orig['Domain_order'] = corr_results_orig['Domain_renamed'].apply(
    lambda x: modality_rank.get(x, 999)
)

# Sort by modality, then by Pearson r within modality
corr_results_by_modality = corr_results_orig.sort_values(
    ['Domain_order', 'Pearson r'],
    ascending=[True, False]
).drop(columns='Domain_order').reset_index(drop=True)

# Rename 'Never eat' column from Diet
corr_results_by_modality['Phenotype'] = corr_results_by_modality['Phenotype'].replace( {'Never eat - I eat all of the above': 'Eat eggs, dairy, wheat, sugar'} )

# Save separate table
corr_results_by_modality.to_excel(
    os.path.join(
        base_path,
        'feature_imp',
        'feature_imp_behav',
        'features',
        'behav_features_g_pred_stack_correlations_sorted_within_modality.xlsx'
    ),
    index=False,
    engine='openpyxl'
)

corr_results_by_modality.to_csv(
    os.path.join(
        base_path,
        'feature_imp',
        'feature_imp_behav',
        'features',
        'behav_features_g_pred_stack_correlations_sorted_within_modality.csv'
    ),
    index=False
)

### Correlation between behavioural features and observed g

*g observed CORR scaled behav features*

In [ ]:
# Combine g-factors predicted from the stacked body model across five folds
g_obs = []

modality_domains = {
    'activity': ['activity_touchscreen', 'accelerometry', 'activity_MET', 'activity_byrecall_i3'],
    'envir': ['localenv'],
    'lifestyle': ['diet', 'electronic_device_use', 'alcohol', 'sun_i0', 'sexual_factors', 'smoking'],
    'sleep': ['sleep_ac']
}

for fold in folds:
    try:
        obs_path = os.path.join(
            base_path,
            'cognition',
            'folds',
            f'fold_{fold}',
            'g',
            f'g_test_with_id_{fold}.csv'
        )
        df = pd.read_csv(obs_path)
        g_obs.append(df)
    except Exception as e:
        print(f"Error loading stacked g-factor for fold {fold}: {str(e)}")

# Combine stacked predictions
g_obs = pd.concat(g_obs, axis=0, ignore_index=True)

# Pool scaled features and merge with eid from pre-scaled files
feature_to_modality = {}
all_features = []

for modality in modalities_behav:
    modality_features = []

    for fold in folds:
        try:
            for domain, modality_list in modality_domains.items():
                if modality in modality_list:
                    scaled_path = os.path.join(
                        base_path, domain, 'folds', f'fold_{fold}', 'scaling',
                        f'{modality}_test_scaled_fold_{fold}.csv'
                    )
                    prescaled_path = os.path.join(
                        base_path, domain, 'folds', f'fold_{fold}', 'suppl',
                        f'{modality}_test_feat_targ_fold_{fold}.csv'
                    )

            scaled_df = pd.read_csv(scaled_path)
            prescaled_df = pd.read_csv(prescaled_path)

            # Map features to domains
            for feature in scaled_df.columns:
                feature_to_modality[feature] = modality

            merged_df = pd.concat([prescaled_df[['eid']], scaled_df], axis=1)
            modality_features.append(merged_df)

        except Exception as e:
            print(f"Error processing {modality}, fold {fold}: {str(e)}")

    if modality_features:
        combined_modality = pd.concat(modality_features, axis=0, ignore_index=True)
        all_features.append(combined_modality)
        
# Merge all modalities column-wise on 'eid'
features_all_modalities = all_features[0]
for df in all_features[1:]:
    features_all_modalities = pd.merge(features_all_modalities, df, on='eid', how='outer')

print(f"Total columns in features_all_modalities: {len(features_all_modalities.columns)}")

# Merge features with stacked g-factor predictions on 'eid'
combined = pd.merge(features_all_modalities, g_obs[['eid', 'g']], on='eid', how='inner')

# Save the final combined DataFrame
output_path = os.path.join(base_path, 'feature_imp', 'feature_imp_behav', 'features', f'behav_features_g_obs_combined.csv')
combined.to_csv(output_path, index=False)

print(f"Final combined shape: {combined.shape}")

In [ ]:
# Compute correlations with observed g-factor
correlations = []
for col in combined.columns:
    if col not in ['eid', 'g']:
        try:
            x = combined['g']
            y = combined[col]
            mask = ~x.isna() & ~y.isna()
            if mask.sum() >= 2:
                r, p = stats.pearsonr(x[mask], y[mask])
            else:
                r, p = np.nan, np.nan

            correlations.append({
                'Phenotype': col,
                'Pearson r': r,
                'p_raw': p      # numeric p-value
            })

        except Exception as e:
            print(f"Error correlating {col}: {str(e)}")
            correlations.append({'Phenotype': col, 'Pearson r': np.nan, 'p_raw': np.nan})

# Convert to DataFrame
corr_results_orig = pd.DataFrame(correlations).reset_index(drop=True)

# Bonferroni correction
n_tests = corr_results_orig['p_raw'].notna().sum()
corr_results_orig['p_bonf'] = corr_results_orig['p_raw'] * n_tests
corr_results_orig['p_bonf'] = corr_results_orig['p_bonf'].clip(upper=1)

# Format
corr_results_orig['p_raw_formatted'] = corr_results_orig['p_raw'].apply(
    lambda p: f"{p:.2e}" if pd.notna(p) else ""
)

# Bonferroni p-values thresholded
corr_results_orig['Bonferroni-corrected p-value'] = corr_results_orig['p_bonf'].apply(
    lambda p: "<0.0001" if (pd.notna(p) and p < 0.0001)
              else (f"{p:.3f}" if pd.notna(p) else "")
)

# Sorted version (Pearson r)
corr_results_pooled_sorted = corr_results_orig.sort_values(
    'Pearson r', ascending=False
).reset_index(drop=True)


# Save to CSV
corr_results_orig.to_csv(os.path.join(base_path, 'feature_imp', 'feature_imp_behav', 'features', 'behav_features_g_obs_correlations_not_sorted.csv'), index=False)
corr_results_pooled_sorted.to_csv(os.path.join(base_path, 'feature_imp', 'feature_imp_behav', 'features', 'behav_features_g_obs_correlations_pooled_sorted.csv'), index=False)

# Apply renaming function before saving to Excel
corr_results_orig['Phenotype'] = corr_results_orig['Phenotype'].apply(
    lambda x: clean_modality_name(x, prefix_to_remove='g_pred_test_')
)

# Save to Excel
corr_results_orig.to_excel(
    os.path.join(base_path, 'feature_imp', 'feature_imp_behav', 'features', 'behav_features_g_obs_correlations_not_sorted.xlsx'),
    index=False,
    engine='openpyxl'
)

corr_results_pooled_sorted.to_excel(
    os.path.join(base_path, 'feature_imp', 'feature_imp_behav', 'features', 'behav_features_g_obs_correlations_pooled_sorted.xlsx'),
    index=False,
    engine='openpyxl'
)

print("Correlation analysis complete. Results saved to CSV and Excel.")

In [ ]:
# Sort correlations within each domain (modality)
corr_results_orig['Domain'] = corr_results_orig['Phenotype'].apply(
    lambda x: feature_to_modality.get(x, 'Unknown')
)

# Map domains
modality_display = {
'activity_touchscreen' :'Daily activity',
'accelerometry':'Accelerometry',
'activity_MET':'Metabolic equivalent of task',
'activity_byrecall_i3':'Yesterday activity',
'sleep_ac':'Sleep',
'localenv':'Local environment',
'diet':'Diet',
'electronic_device_use':'Electronic device use',
'alcohol':'Alcohol',
'sun_i0':'Sun exposure',
'sexual_factors':'Sexual factors',
'smoking':'Smoking',
}

# Add display name column
corr_results_orig['Domain_renamed'] = corr_results_orig['Domain'].apply(
    lambda x: modality_display.get(x, 'Unknown')
)

# Desired order of display names
modality_order = [
    'Daily activity',
    'Diet',
    'Accelerometry',
    'Sun exposure',
    'Metabolic equivalent of task',
    'Local environment',
    'Sleep',
    'Alcohol',
    'Smoking',
    'Electronic device use',
    'Sexual factors',
    'Yesterday activity'
]

# Create rank for sorting
modality_rank = {name: i for i, name in enumerate(modality_order)}

# Add rank column
corr_results_orig['Domain_order'] = corr_results_orig['Domain_renamed'].apply(
    lambda x: modality_rank.get(x, 999)
)

# Sort by modality, then by Pearson r within modality
corr_results_by_modality = corr_results_orig.sort_values(
    ['Domain_order', 'Pearson r'],
    ascending=[True, False]
).drop(columns='Domain_order').reset_index(drop=True)

# Rename 'Never eat' column from Diet
corr_results_by_modality['Phenotype'] = corr_results_by_modality['Phenotype'].replace( {'Never eat - I eat all of the above': 'Eat eggs, dairy, wheat, sugar'} )

# Save separate table
corr_results_by_modality.to_excel(
    os.path.join(
        base_path,
        'feature_imp',
        'feature_imp_behav',
        'features',
        'behav_features_g_obs_correlations_sorted_within_modality.xlsx'
    ),
    index=False,
    engine='openpyxl'
)

corr_results_by_modality.to_csv(
    os.path.join(
        base_path,
        'feature_imp',
        'feature_imp_behav',
        'features',
        'behav_features_g_obs_correlations_sorted_within_modality.csv'
    ),
    index=False
)